# 09 · Original DA3 vs all enhancements
Reads the standard result records (`da3.compare`) of every run: the committed ones in
`experiments/results/` plus any attached notebook outputs (`/kaggle/input/*/…/results`). Runs are
only compared when they share weights, datasets, scenes and frame sampling. CPU is enough
(**Accelerator = None**, **Internet = on** for the clone).

In [ ]:
# --- repo setup (identical in every notebook) ---------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess, sys
os.chdir("/kaggle/working")
url = REPO_URL
try:  # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
!git log -1 --format="commit %H  %cd"

In [ ]:
import glob
RESULT_DIRS = ["experiments/results"] + sorted(set(glob.glob("/kaggle/input/*/results") +
                                                  glob.glob("/kaggle/input/*/*/results")))
print(RESULT_DIRS)
dirs = " ".join(RESULT_DIRS)
!python scripts/compare_results.py --dirs {dirs} --out /kaggle/working/comparison.md > /dev/null

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("/kaggle/working/comparison.md").read()))

In [ ]:
import matplotlib.pyplot as plt
from da3.compare import load_records
from da3.compare.plots import sec6_delta_bars
recs = load_records(*RESULT_DIRS)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, lab in zip(axes, ["AUC@3", "AUC@30", "F1 w/o p."]):
    sec6_delta_bars(recs, lab, ax=ax)
plt.tight_layout(); plt.show()